# Ultra Small Language Models benchmarked

## Ollama Installation alongwith ZStandard

In [1]:
!nvidia-smi

Wed May 20 03:53:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!apt install

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.


In [3]:
!apt-get update && apt install zstd -y
!pip install ollama
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-se

## Setup and Begin Running Ollama Server

In [4]:
import os
import threading
import subprocess
import ollama

def run_ollama():
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    subprocess.Popen(["ollama", "serve"])

threading.Thread(target=run_ollama).start()


### Decorator to log function calls

In [5]:
# Decorator function to help in logging the tools called by the Models

import functools

logs = {}

log_count = 0

def logCount_Incrementor(log_count):
    """Increments the Log Count by 1"""
    log_count += 1
    return log_count

#Tool Use tracker through decorators
def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        #Log before function runs
        logs.update({logCount_Incrementor(log_count) :f"Log Calling {func.__name__} with args: f{args} and  kwargs:{kwargs}"})

        return func(*args, **kwargs)

    return wrapper

### Develop 5 tools

In [6]:
import json
import random

#Get Machine State
@log_call
def get_machine_state(**kwargs)-> str:
    """Get Current Machine State"""
    return json.dumps({
        'State':'Active'
    })

#Get Weather
@log_call
def get_weather(location: str, **kwargs) ->str:
    """Get the weather based on the location provided by the user"""
    return json.dumps({
        'location': location,
        'temperature': 36
    })

#Validate Switch
@log_call
def validate_switch(**kwargs):
    """Check the status of the switch"""
    return json.dumps({
        "Status":"Switch Status: ON"
        })

#Setup Meeting
meetings = {}
@log_call
def setup_meeting(persons: int, time: str, **kwargs):
    """Schedule a meeting"""
    meetings.update({'Time': time, "Persons":persons})
    return json.dumps({
        'message':f"Meeting has been scheduled as requested: Persons: {persons} Time: {time}"
        })

#Get Population
@log_call
def get_population(**kwargs):
    """Get population data"""
    return json.dumps({
        'population':100000
    })

# Available tools
available_tools = {
        'get_machine_state' :   get_machine_state,
        'get_weather'       :   get_weather,
        'validate_switch'   :   validate_switch,
        'setup_meeting'     :   setup_meeting,
        'get_population'    :   get_population,
}

===========================================================================================================


# **Actual Benchmarking**

In [7]:
ROLE = 'user'

#Experimental Prompts

P1  = "What is the Current Machine state?"
P2  = "What is the weather like in Mauritius?"
P3  = "Have I turned off the Switch?"
P4  = "Do schedule a meeting for 8 persons at 12 PM."
P5  = "My Machine state is Active, what's yours?"
P6  = "Whats the population?"
P7  = "Dont check the machine state, just tell me if the switch is ON?"
P8  = "Blue Ocean Meeting Bits Dublin"
P9  = "I do not want to schedule a meeting of 2 persons at 6pm."
P10 = "The population of India is 150000000000. Whats the weather like in Delhi?"
P11 = "List the tools you have access to."
P12 = "I want to visit Kolkata, anything I should know?"
P13 = "How could he have set up a meeting for 6 people at 4pm without notifying me?"
P14 = "Tommorrow is a holiday, I shouldn't set up a meeting, right?"
P15 = "Why is population so hard to calculate?"
P16 = ""
P17 = ""
P18 = ""
P19 = ""
P20 = ""
P21 = ""
P22 = ""
P23 = ""
P24 = ""
P25 = ""
P26 = ""
P27 = ""
P28 = ""
P29 = ""
P30 = ""

prompts = [P1, P2, P3, P4, P5, P6, P7,P8, P9, P10, P11, P12, P13, P14, P15]

prerequisite = ""

## Message Generator
def messageGen(prompt: str):
    return {'role':ROLE,
            'content': prompt}


### Model 1: qwen2.5:0.5b

In [8]:
!ollama pull qwen2.5:0.5b

In [9]:
# Initial Test Runs with qwen2.5:0.5b
from ollama import chat
import ollama

for prompt in prompts:
    print(f"\n--- Current Prompt: {prompt} ---")

    messages = [messageGen(prompt)]
    response = chat(
        model='qwen2.5:0.5b',
        messages = messages,
        tools = [get_machine_state, get_weather, validate_switch, setup_meeting, get_population]
    )

    if response.get('message') and response['message'].get('tool_calls'):
        # Add assistant's tool call intent to message history
        messages.append(response['message'])

        for tool_call in response['message']['tool_calls']:
            function_to_call = tool_call['function']['name']
            function_args = tool_call['function']['arguments']

            if function_to_call in available_tools:
                print(f"[Tool Calling]: {function_to_call} with {function_args}")
                tool_output = available_tools[function_to_call](**function_args)

                # Feed the tool result back using the 'content' key
                messages.append({
                    'role': 'tool',
                    'content': tool_output
                })

        # Final call to get the text response based on tool results
        final_response = ollama.chat(model = 'qwen2.5:0.5b', messages = messages)
        print("Assistant Response:", final_response['message']['content'])
    else:
        print("Assistant Response:", response['message']['content'])

print("\nExecution Logs:", logs)


--- Current Prompt: What is the Current Machine state? ---
[Tool Calling]: get_machine_state with {'kwargs': ''}
Assistant Response: The machine is currently in an active state. It's running and can be accessed through the CLI interface or the web UI.

--- Current Prompt: What is the weather like in Mauritius? ---
[Tool Calling]: get_weather with {'location': 'Mauritius'}
Assistant Response: The weather in Mauritius is very pleasant with a temperature of 36 degrees Celsius. It's a hot and sunny place, making it perfect for outdoor activities such as sunbathing, hiking, and engaging in other forms of leisure and recreation. If you plan on visiting, don't forget to pack your sunscreen and a light jacket!

--- Current Prompt: Have I turned off the Switch? ---
[Tool Calling]: get_machine_state with {'kwargs': ''}
Assistant Response: The switch is currently active. If you want to turn it off, you can use the `get_machine_state` function again and provide a reason or explanation for turning

In [10]:
print(logs)

{1: "Log Calling get_weather with args: f() and  kwargs:{'location': 'Delhi'}"}


### Model2: qwen3: 0.6b

In [11]:
!ollama pull qwen3:0.6b

In [12]:
# Initial Test Runs with qwen2.5:0.5b
from ollama import chat
import ollama

for prompt in prompts:
    print(f"\n--- Current Prompt: {prompt} ---")

    messages = [messageGen(prompt)]
    response = chat(
        model='qwen3:0.6b',
        messages = messages,
        tools = [get_machine_state, get_weather, validate_switch, setup_meeting, get_population]
    )

    if response.get('message') and response['message'].get('tool_calls'):
        # Add assistant's tool call intent to message history
        messages.append(response['message'])

        for tool_call in response['message']['tool_calls']:
            function_to_call = tool_call['function']['name']
            function_args = tool_call['function']['arguments']

            if function_to_call in available_tools:
                print(f"[Tool Calling]: {function_to_call} with {function_args}")
                tool_output = available_tools[function_to_call](**function_args)

                # Feed the tool result back using the 'content' key
                messages.append({
                    'role': 'tool',
                    'content': tool_output
                })

        # Final call to get the text response based on tool results
        final_response = ollama.chat(model = 'qwen3:0.6b', messages = messages)
        print("Assistant Response:", final_response['message']['content'])
    else:
        print("Assistant Response:", response['message']['content'])

print("\nExecution Logs:", logs)


--- Current Prompt: What is the Current Machine state? ---
Assistant Response: 

--- Current Prompt: What is the weather like in Mauritius? ---
[Tool Calling]: get_weather with {'location': 'Mauritius'}
Assistant Response: The weather in Mauritius is currently **36°C**. Let me know if you'd like more details!

--- Current Prompt: Have I turned off the Switch? ---
Assistant Response: I don't have access to the current state of the switch. To check its status, I would need to call the validate_switch function with the appropriate parameters. Could you please provide the switch's status or details?

--- Current Prompt: Do schedule a meeting for 8 persons at 12 PM. ---
[Tool Calling]: setup_meeting with {'persons': 8, 'time': '12 PM'}
Assistant Response: The meeting has been scheduled as requested:  
**Persons: 8** at **12 PM**.

--- Current Prompt: My Machine state is Active, what's yours? ---
[Tool Calling]: get_machine_state with {'kwargs': 'Active'}
Assistant Response: Both your machi

### Model3: qwen3:4b

In [13]:
!ollama pull qwen3:4b

In [14]:
# Initial Test Runs with qwen3:4b
from ollama import chat
import ollama

for prompt in prompts:
    print(f"\n--- Current Prompt: {prompt} ---")

    messages = [messageGen(prompt)]
    response = chat(
        model='qwen3:4b',
        messages = messages,
        tools = [get_machine_state, get_weather, validate_switch, setup_meeting, get_population]
    )

    if response.get('message') and response['message'].get('tool_calls'):
        # Add assistant's tool call intent to message history
        messages.append(response['message'])

        for tool_call in response['message']['tool_calls']:
            function_to_call = tool_call['function']['name']
            function_args = tool_call['function']['arguments']

            if function_to_call in available_tools:
                print(f"[Tool Calling]: {function_to_call} with {function_args}")
                tool_output = available_tools[function_to_call](**function_args)

                # Feed the tool result back using the 'content' key
                messages.append({
                    'role': 'tool',
                    'content': tool_output
                })

        # Final call to get the text response based on tool results
        final_response = ollama.chat(model = 'qwen3:4b', messages = messages)
        print("Assistant Response:", final_response['message']['content'])
    else:
        print("Assistant Response:", response['message']['content'])

print("\nExecution Logs:", logs)


--- Current Prompt: What is the Current Machine state? ---
[Tool Calling]: get_machine_state with {'kwargs': ''}
Assistant Response: The current machine state is Active.

--- Current Prompt: What is the weather like in Mauritius? ---
[Tool Calling]: get_weather with {'location': 'Mauritius'}
Assistant Response: The current temperature in Mauritius is **36°C**. As a tropical island nation, Mauritius typically experiences warm and humid weather year-round, with temperatures ranging from 25°C to 32°C during the day. The climate is influenced by its proximity to the Indian Ocean and the monsoon seasons. Let me know if you'd like more specific details! 😊

--- Current Prompt: Have I turned off the Switch? ---
[Tool Calling]: validate_switch with {'kwargs': ''}
Assistant Response: No, the Switch is currently **ON**. It hasn't been turned off.

--- Current Prompt: Do schedule a meeting for 8 persons at 12 PM. ---
[Tool Calling]: setup_meeting with {'persons': 8, 'time': '12 PM'}
Assistant Res

### Model4: ministral-3:3b

In [15]:
!ollama pull ministral-3:3b

In [16]:
# Initial Test Runs with phi4-mini:3.8b
from ollama import chat
import ollama

for prompt in prompts:
    print(f"\n--- Current Prompt: {prompt} ---")

    messages = [messageGen(prompt)]
    response = chat(
        model='ministral-3:3b',
        messages = messages,
        tools = [get_machine_state, get_weather, validate_switch, setup_meeting, get_population]
    )

    if response.get('message') and response['message'].get('tool_calls'):
        # Add assistant's tool call intent to message history
        messages.append(response['message'])

        for tool_call in response['message']['tool_calls']:
            function_to_call = tool_call['function']['name']
            function_args = tool_call['function']['arguments']

            if function_to_call in available_tools:
                print(f"[Tool Calling]: {function_to_call} with {function_args}")
                tool_output = available_tools[function_to_call](**function_args)

                # Feed the tool result back using the 'content' key
                messages.append({
                    'role': 'tool',
                    'content': tool_output
                })

        # Final call to get the text response based on tool results
        final_response = ollama.chat(model = 'ministral-3:3b', messages = messages)
        print("Assistant Response:", final_response['message']['content'])
    else:
        print("Assistant Response:", response['message']['content'])

print("\nExecution Logs:", logs)


--- Current Prompt: What is the Current Machine state? ---
[Tool Calling]: get_machine_state with {'kwargs': '{}'}
Assistant Response: Le Chat est actuellement dans un état **actif** et prêt à répondre à vos questions ou à traiter vos demandes. 😊

Si vous avez besoin d'informations spécifiques ou d'une assistance pour autre chose, n'hésitez pas à me le faire savoir !

--- Current Prompt: What is the weather like in Mauritius? ---
Assistant Response: Could you please specify the city or region in Mauritius you'd like to know the weather for? For example, Port Louis, Réunion Island (which is part of Mauritius), or another location? This will help me provide the most accurate weather forecast.

--- Current Prompt: Have I turned off the Switch? ---
Assistant Response: Could you clarify what you mean by "the Switch"? Are you referring to a specific electronic device, appliance, or system (e.g., a smart home switch, a server, a light fixture, etc.)? If you provide more context, I can help c

### Model5: lfm2.5-thinking:1.2b

In [17]:
!ollama pull lfm2.5-thinking:1.2b

In [18]:
# Initial Test Runs with lfm2.5-thinking:1.2b
from ollama import chat
import ollama

for prompt in prompts:
    print(f"\n--- Current Prompt: {prompt} ---")

    messages = [messageGen(prompt)]
    response = chat(
        model='lfm2.5-thinking:1.2b',
        messages = messages,
        tools = [get_machine_state, get_weather, validate_switch, setup_meeting, get_population]
    )

    if response.get('message') and response['message'].get('tool_calls'):
        # Add assistant's tool call intent to message history
        messages.append(response['message'])

        for tool_call in response['message']['tool_calls']:
            function_to_call = tool_call['function']['name']
            function_args = tool_call['function']['arguments']

            if function_to_call in available_tools:
                print(f"[Tool Calling]: {function_to_call} with {function_args}")
                tool_output = available_tools[function_to_call](**function_args)

                # Feed the tool result back using the 'content' key
                messages.append({
                    'role': 'tool',
                    'content': tool_output
                })

        # Final call to get the text response based on tool results
        final_response = ollama.chat(model = 'lfm2.5-thinking:1.2b', messages = messages)
        print("Assistant Response:", final_response['message']['content'])
    else:
        print("Assistant Response:", response['message']['content'])

print("\nExecution Logs:", logs)


--- Current Prompt: What is the Current Machine state? ---
[Tool Calling]: get_machine_state with {'kwargs': ''}
Assistant Response: The current machine state is \boxed{Active}.

--- Current Prompt: What is the weather like in Mauritius? ---
[Tool Calling]: get_weather with {'location': 'Mauritius', 'kwargs': ''}
Assistant Response: The current temperature in Mauritius is approximately **36°C** (which is about 97.8°F), reflecting its tropical climate. Mauritius typically experiences warm, humid conditions year-round, with high humidity and occasional rainfall, especially during the rainy season (November to April). While specific conditions vary by season and location within the island, the weather often includes a mix of sun and showers, making it a tropical destination with distinct seasonal variations. Let me know if you'd like further details!

--- Current Prompt: Have I turned off the Switch? ---
[Tool Calling]: get_machine_state with {'keywords': 'Switch'}
Assistant Response: Th